#General Ledger data cleaning

## Import dependency and data

In [0]:
import pyspark.sql.functions as F

In [0]:
df_ledger_raw = spark.table("bronze.azure_blob_storage.general_ledgers")

In [0]:
df_ledger_raw.display()

##Removing columns inserted by fivetron

In [0]:
df_ledger=df_ledger_raw.drop('_file','_line','_modified','_fivetran_synced')

In [0]:
df_ledger.display()

## Type casting

In [0]:
from pyspark.sql.functions import col, to_date, col
df_ledger = df_ledger.withColumn("entry_date", to_date(col("entry_date"), "dd-MM-yyyy HH:mm")) \
    .withColumn("posting_date", to_date(col("posting_date"), "dd-MM-yyyy HH:mm"))

In [0]:
display(df_ledger)

In [0]:
df_ledger=df_ledger.withColumn("gl_id",col('gl_id').cast('int'))

## handling Null values

In [0]:
from pyspark.sql import functions as F
for column in df_ledger.columns:
    null_count = df_ledger.filter(F.col(column).isNull()).count()
    print(f"{column}: {null_count}")

## Checking for Duplicates

In [0]:
if df_ledger.count()>df_ledger.dropDuplicates().count():
  df_ledger=df_ledger.dropDuplicates

In [0]:
df_ledger.display()


In [0]:
for col_name, data_type in df_ledger.dtypes:
    if data_type == 'bigint':
        df_ledger = df_ledger.withColumn(col_name, df_ledger[col_name].cast('int'))

In [0]:
df_ledger.display()

## Handling null values

In [0]:
for column in df_ledger.columns:
    null_count = df_ledger.filter(F.col(column).isNull()).count()
    print(f"{column}: {null_count}")

## Handling Duplicates

In [0]:
if df_ledger.count() > df_ledger.dropDuplicates().count():
    df_ledger = df_ledger.dropDuplicates()
df_ledger.display()

## Writing to silver

In [0]:
df_ledger.write.mode("overwrite").saveAsTable("silver.transformation.general_ledgers")